<a href="https://colab.research.google.com/github/temahm/AiCon/blob/main/JobDescriptionBiasAssessmentV4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JOB DESCRIPTION BIAS ASSESSMENT

**This version uses weights scale for bias keywords**

Cell 1. Imports

In [ ]:
!pip -q install -U google-genai

import re
import json
import math
import pandas as pd
from google.colab import userdata
from google import genai

Cell 2. Bias Keywords dictionalry and its assigned weights

In [ ]:
# This lexicon is a curated heuristic, not an official legal dictionary.
# Source basis:
# - Research on gender-coded wording in job advertisements
# - EEOC protected-category guidance
# - EEOC national-origin guidance
# - ADA / EEOC essential-functions and accommodation guidance

BIAS_LEXICON_NOTES = {
    "Gender Bias": "Research-inspired cues, especially gender-coded wording in job ads.",
    "Age Bias": "Compliance-risk cues related to age-coded hiring language.",
    "Racial/Ethnic Bias": "Compliance-risk cues related to national origin, accent, and linguistic characteristics.",
    "Disability/ADA Bias": "Physical/medical requirements that may be problematic unless they are essential job functions.",
    "Socioeconomic Bias": "Class/elitism screening cues and access barriers.",
    "Exclusionary Language": "Broad gatekeeping or overly rigid requirements.",
    "Regulatory Risk": "Potentially sensitive protected-trait or pre-employment inquiry language."
}

# Weight scale:
# 1 = weak cue / high false-positive
# 2 = mild concern
# 3 = moderate concern
# 4 = strong concern
# 5 = high-risk / likely legal or fairness concern depending on context

bias_keywords = {
    "Gender Bias": {
        "aggressive": 2,
        "ambitious": 2,
        "assertive": 2,
        "competitive": 2,
        "confident": 1,
        "decisive": 2,
        "dominant": 3,
        "driven": 1,
        "fearless": 2,
        "independent": 1,
        "leader": 1,
        "leadership": 1,
        "rockstar": 3,
        "ninja": 3,
        "killer instinct": 4,
        "strong": 1,
        "superior": 2,
        "headstrong": 3,
        "he": 1,
        "she": 1,
        "his": 1,
        "her": 1,
        "gentleman": 4,
        "lady": 4,
        "manpower": 4,
        "salesman": 4,
        "foreman": 4,
        "waitress": 4,
        "actress": 3,
        "mankind": 3
    },

    "Age Bias": {
        "young": 3,
        "youthful": 4,
        "energetic": 2,
        "digital native": 4,
        "recent graduate": 4,
        "new grad": 3,
        "fresh talent": 4,
        "early career only": 4,
        "mature": 2,
        "seasoned": 2,
        "elderly": 5,
        "retired": 4,
        "overqualified": 2,
        "high potential youth": 5,
        "young blood": 5,
        "under 30": 5,
        "under 35": 5
    },

    "Racial/Ethnic Bias": {
        "native speaker": 4,
        "native english": 4,
        "english native": 4,
        "cultural fit": 2,
        "fits our culture": 2,
        "american-born": 5,
        "no foreign accent": 5,
        "local only": 3,
        "must be us-born": 5
    },

    "Disability/ADA Bias": {
        "able-bodied": 5,
        "physically fit": 3,
        "perfect health": 5,
        "healthy": 4,
        "must stand": 2,
        "stand for long periods": 3,
        "must walk": 2,
        "walk for extended periods": 3,
        "lift heavy objects": 3,
        "lift 50 pounds": 4,
        "physically demanding": 2,
        "independent mobility": 4,
        "good hearing": 3,
        "excellent hearing": 3,
        "good vision": 3,
        "excellent vision": 3,
        "sight": 1,
        "hearing": 1,
        "medical history": 5,
        "disability status": 5
    },

    "Socioeconomic Bias": {
        "prestigious university": 3,
        "top-tier university": 3,
        "elite background": 4,
        "ivy league": 3,
        "unpaid internship": 3,
        "car ownership": 3,
        "must own a car": 3,
        "polished background": 2,
        "well-spoken background": 2
    },

    "Exclusionary Language": {
        "must have": 1,
        "only": 2,
        "everyone knows": 3,
        "we expect": 1,
        "exclusive": 2,
        "no exceptions": 3,
        "mandatory pedigree": 4,
        "required pedigree": 4,
        "must already know": 2,
        "must be an expert from day one": 3
    },

    "Regulatory Risk": {
        "date of birth": 4,
        "married": 4,
        "single": 4,
        "family status": 4,
        "children": 3,
        "pregnant": 5,
        "nationality": 5,
        "citizenship required": 3,
        "religion": 5,
        "religions": 5,
        "christian": 5,
        "muslim": 5,
        "jewish": 5,
        "criminal record": 3,
        "background checks": 2
    }
}

print("Weighted bias lexicon initialized.")
print("Categories:", list(bias_keywords.keys()))

Cell 3. Paste the job description. Doble-Enter to finish.

In [ ]:
print("Paste the job description below. Press Enter on an empty line TWICE to finish.\n")

lines = []
empty_count = 0

while True:
    line = input()
    if line.strip() == "":
        empty_count += 1
        if empty_count >= 2:
            break
    else:
        empty_count = 0
        lines.append(line)

job_description_text = "\n".join(lines).strip()

print("\nJob description received.")
print("Character count:", len(job_description_text))
print("\nPreview:\n")
print(job_description_text[:700])

Cell 4 — Gemini setup and model selection

In [ ]:
API_KEY = userdata.get("GEMINI_API_KEY")
if not API_KEY:
    raise RuntimeError("Please add your Gemini API key to Colab Secrets as GEMINI_API_KEY")

client = genai.Client(api_key=API_KEY)

def pick_generate_content_model(client):
    preferred = [
        "models/gemini-2.5-flash",
        "models/gemini-2.0-flash",
        "models/gemini-1.5-flash",
        "models/gemini-1.5-pro",
    ]

    available = []
    for m in client.models.list():
        name = getattr(m, "name", None)
        methods = getattr(m, "supported_actions", None) or getattr(m, "supported_methods", None)
        methods_str = " ".join(methods) if isinstance(methods, (list, tuple)) else str(methods)
        if name and ("generateContent" in methods_str or "generate_content" in methods_str):
            available.append(name)

    for p in preferred:
        if p in available:
            return p

    return available[0] if available else None

MODEL_NAME = pick_generate_content_model(client)
if not MODEL_NAME:
    raise RuntimeError("No available Gemini model supports generateContent for this API key.")

print("Using model:", MODEL_NAME)

def llm(prompt: str) -> str:
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt
    )
    return (response.text or "").strip()

Cell 5 — Helpers for matching and JSON parsing

In [ ]:
def make_keyword_pattern(keyword: str) -> re.Pattern:
    """
    Robust matching:
    - single-word phrases matched as whole tokens
    - multi-word phrases allow flexible whitespace between words
    """
    kw = keyword.strip()
    tokens = [re.escape(t) for t in kw.split()]
    if len(tokens) == 1:
        pattern = r'(?<!\w)' + tokens[0] + r'(?!\w)'
    else:
        pattern = r'(?<!\w)' + r'\s+'.join(tokens) + r'(?!\w)'
    return re.compile(pattern, flags=re.IGNORECASE)

def get_context(text: str, start: int, end: int, window: int = 140) -> str:
    cs = max(0, start - window)
    ce = min(len(text), end + window)
    return text[cs:ce].strip()

def extract_json(text: str):
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        return None
    try:
        return json.loads(text[start:end+1])
    except Exception:
        return None

def safe_int(value, default=1):
    try:
        return int(value)
    except Exception:
        return default

Cell 6. Weighted keyword scan

In [ ]:
raw_hits = []

for category, weighted_terms in bias_keywords.items():
    for keyword, weight in weighted_terms.items():
        rx = make_keyword_pattern(keyword)
        for m in rx.finditer(job_description_text):
            raw_hits.append({
                "source": "keyword_scan",
                "category_hint": category,
                "keyword": keyword,
                "keyword_weight": weight,
                "phrase": m.group(0),
                "context": get_context(job_description_text, m.start(), m.end())
            })

print("Raw keyword hits found:", len(raw_hits))

for h in raw_hits[:15]:
    print(f"- [{h['category_hint']}] '{h['phrase']}' | weight={h['keyword_weight']} | keyword='{h['keyword']}'")

Cell 7. One-pass LLM discovery

In [ ]:
discovery_prompt = f"""
You are auditing a job description for potentially biased, exclusionary, or legally risky language.

Instructions:
1. Identify phrases that may discourage protected groups or create fairness/compliance concerns.
2. Consider categories such as gender, age, race/ethnicity, national origin, disability/ADA, religion, family status, socioeconomic access, and broad exclusionary language.
3. Be conservative but useful: do not flag everything, only phrases with a plausible fairness or compliance concern.
4. Return JSON ONLY in the exact schema below.

JSON schema:
{{
  "findings": [
    {{
      "phrase": "exact phrase from the job description",
      "category": "Gender Bias | Age Bias | Racial/Ethnic Bias | Disability/ADA Bias | Socioeconomic Bias | Exclusionary Language | Regulatory Risk | Other",
      "severity": 1,
      "reason": "short reason",
      "suggested_rewrite": "neutral alternative wording"
    }}
  ]
}}

Job description:
\"\"\"{job_description_text}\"\"\"
"""

discovery_text = llm(discovery_prompt)
print(discovery_text[:1500])

Cell 8. Merge keyword hits with LLM discoveries

In [ ]:
discovery = extract_json(discovery_text)
llm_findings = discovery.get("findings", []) if discovery else []

print("LLM discovery findings:", len(llm_findings))

merged = []

for h in raw_hits:
    merged.append({
        "source": h["source"],
        "category_hint": h["category_hint"],
        "phrase": h["phrase"],
        "context": h["context"],
        "keyword_weight": h["keyword_weight"],
        "severity": None,
        "reason": None,
        "suggested_rewrite": None
    })

for f in llm_findings:
    merged.append({
        "source": "llm_discovery",
        "category_hint": f.get("category", "Other"),
        "phrase": (f.get("phrase") or "").strip(),
        "context": None,
        "keyword_weight": None,
        "severity": f.get("severity"),
        "reason": f.get("reason"),
        "suggested_rewrite": f.get("suggested_rewrite")
    })

cleaned = []
seen = set()

for item in merged:
    phrase = (item["phrase"] or "").strip()
    if not phrase:
        continue
    key = phrase.lower()
    if key in seen:
        continue
    seen.add(key)
    cleaned.append(item)

print("Merged unique candidates:", len(cleaned))
for c in cleaned[:15]:
    print("-", c["phrase"], "|", c["category_hint"])

Cell 9 — LLM validation per candidate

In [ ]:
def validate_candidate(phrase, category_hint, context_hint, keyword_weight=None):
    context_text = context_hint or job_description_text

    prompt = f"""
You are validating a candidate fairness/compliance issue in a job description.

Return JSON ONLY:
{{
  "is_issue": true,
  "category": "Gender Bias | Age Bias | Racial/Ethnic Bias | Disability/ADA Bias | Socioeconomic Bias | Exclusionary Language | Regulatory Risk | Other",
  "severity": 1,
  "reason": "short reason",
  "suggested_rewrite": "neutral rewrite"
}}

Rules:
- If the phrase is not a real issue in context, return "is_issue": false.
- Severity must be 1 to 5.
- Keep the reason short and specific.
- If a physical requirement appears legitimate only if essential to the role, reflect that in your judgment.

Phrase: "{phrase}"
Category hint: "{category_hint}"
Keyword weight hint: "{keyword_weight}"

Context:
\"\"\"{context_text}\"\"\"
"""
    response_text = llm(prompt)
    data = extract_json(response_text)
    return data

validated = []

for item in cleaned:
    data = validate_candidate(
        phrase=item["phrase"],
        category_hint=item["category_hint"],
        context_hint=item["context"],
        keyword_weight=item["keyword_weight"]
    )

    if not data:
        continue

    if data.get("is_issue") is True:
        item["category"] = data.get("category", item["category_hint"])
        item["severity"] = safe_int(data.get("severity"), default=1)
        item["reason"] = data.get("reason", "")
        item["suggested_rewrite"] = data.get("suggested_rewrite", "")
        validated.append(item)

print("Validated issues kept:", len(validated))
for v in validated[:15]:
    print(f"- [{v['category']}] '{v['phrase']}' | severity={v['severity']}")

Score, summary, and report


In [ ]:
def severity_points(sev):
    sev = safe_int(sev, default=1)
    return {
        1: 4,
        2: 8,
        3: 12,
        4: 16,
        5: 20
    }.get(sev, 4)

category_multiplier = {
    "Regulatory Risk": 1.25,
    "Disability/ADA Bias": 1.25,
    "Racial/Ethnic Bias": 1.15,
    "Age Bias": 1.15,
    "Gender Bias": 1.15,
    "Exclusionary Language": 1.0,
    "Socioeconomic Bias": 1.0,
    "Other": 0.8,
}

score_raw = 0.0

for item in validated:
    sev = safe_int(item.get("severity"), default=1)
    category = item.get("category", item.get("category_hint", "Other"))
    base_points = severity_points(sev)
    mult = category_multiplier.get(category, 1.0)

    # Optional small boost if the original keyword list already marked it as strong
    kw_weight = item.get("keyword_weight")
    kw_boost = 0
    if kw_weight is not None:
        kw_boost = max(0, safe_int(kw_weight, default=1) - 1) * 0.8

    score_raw += (base_points * mult) + kw_boost

bias_score = min(100, round(score_raw, 1))

def score_band(score):
    if score <= 10:
        return "Low"
    elif score <= 30:
        return "Moderate"
    elif score <= 60:
        return "High"
    else:
        return "Very High"

risk_level = score_band(bias_score)

print("=" * 80)
print("BIAS ASSESSMENT REPORT")
print("=" * 80)
print("Bias Risk Score (0–100):", bias_score)
print("Risk Level:", risk_level)
print("Validated Issue Count:", len(validated))
print()

df = pd.DataFrame(validated)

if df.empty:
    print("No validated issues found.")
else:
    report_cols = [
        "category",
        "severity",
        "phrase",
        "reason",
        "suggested_rewrite",
        "source",
        "keyword_weight"
    ]
    df = df[[c for c in report_cols if c in df.columns]]

    print("Summary by category:")
    print(df.groupby("category").size().sort_values(ascending=False))
    print()

    print("Detailed findings:")
    display(df.sort_values(["category", "severity"], ascending=[True, False]))

    print("\nSuggested rewrites:")
    for _, row in df.head(20).iterrows():
        print(f"- '{row['phrase']}' -> {row['suggested_rewrite']}")